# Chương 1 — Từ dữ liệu đến quyết định kinh doanh

Notebook minh hoạ các nội dung: **BI – BA – DSS**, tư duy phân tích dữ liệu,
chu trình ra quyết định của Simon, chi phí sai và lựa chọn phương án.

**Tình huống xuyên suốt:** một doanh nghiệp viễn thông muốn giảm tỷ lệ khách
hàng rời mạng (churn) với ngân sách giữ chân hữu hạn.

> Dữ liệu hoàn toàn mô phỏng phục vụ học tập. Chọn **Runtime → Run all** trên
> Google Colab để chạy toàn bộ notebook.

## Mục tiêu học tập

Sau notebook, người học có thể:

1. Phân biệt đầu ra của BI, BA và DSS.
2. Chuyển vấn đề kinh doanh thành câu hỏi dữ liệu và KPI.
3. Đánh giá một quyết định bằng lợi ích, chi phí và rủi ro — không chỉ bằng độ chính xác.
4. Mô tả bốn pha: nhận diện, thiết kế, lựa chọn, thực hiện và học hỏi.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
plt.style.use("seaborn-v0_8-whitegrid")
RANDOM_SEED = 42

## 1. Tạo dữ liệu mẫu bằng Python

**Grain:** mỗi dòng là một khách hàng tại thời điểm đầu tháng 5/2026. Biến
`churned` chỉ kết quả quan sát cuối tháng. `risk_score` là điểm rủi ro do một
mô hình phân tích cung cấp (giá trị càng cao, nguy cơ rời mạng càng lớn).

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
n = 600
segment = rng.choice(list("ABCD"), n, p=[0.25, 0.30, 0.25, 0.20])
tenure = rng.integers(1, 73, n)
monthly_fee = np.round(rng.normal(280_000, 75_000, n).clip(90_000, 600_000), -3)
complaints = rng.poisson(0.55, n)
late_payment = rng.binomial(1, 0.18, n)
usage_change = np.round(rng.normal(-0.03, 0.18, n), 3)

logit = (-2.8 + 0.9*(segment == "A") + 0.48*complaints
         + 0.75*late_payment - 1.8*usage_change - 0.012*tenure)
risk_score = 1 / (1 + np.exp(-logit))
churned = rng.binomial(1, risk_score)

customers = pd.DataFrame({
    "customer_id": [f"KH{i:04d}" for i in range(1, n+1)],
    "segment": segment, "tenure_months": tenure,
    "monthly_fee": monthly_fee.astype(int), "complaints": complaints,
    "late_payment": late_payment, "usage_change": usage_change,
    "risk_score": risk_score.round(4), "churned": churned
})
customers.head()

In [ ]:
assert customers["customer_id"].is_unique
assert customers["risk_score"].between(0, 1).all()
print(f"Số khách hàng: {len(customers):,}")
print(f"Churn toàn bộ: {customers['churned'].mean():.2%}")

## Bài tập 1 — Nhận diện BI, BA và DSS

Điền `BI`, `BA` hoặc `DSS` vào cột `your_answer` theo **vai trò chính** của
từng đầu ra. Một hệ thống thực tế có thể chồng lấn; ở đây hãy dựa vào câu hỏi
chính mà chức năng trả lời.

In [ ]:
functions = pd.DataFrame({
    "function": [
        "Dashboard churn theo phân khúc", "Ước lượng xác suất churn",
        "Cảnh báo churn vượt 5%", "Giải thích yếu tố liên quan đến churn",
        "Chọn khách nhận ưu đãi trong ngân sách", "Mô phỏng ba gói ưu đãi"
    ],
    "question": [
        "Điều gì đang xảy ra?", "Điều gì có khả năng xảy ra?",
        "Điểm nào cần chú ý?", "Vì sao/điều gì liên quan?",
        "Nên chọn phương án nào?", "Nếu chọn phương án thì điều gì xảy ra?"
    ],
    "your_answer": [""] * 6
})
functions

<details><summary><b>Gợi ý đáp án</b></summary>

Theo thứ tự: **BI, BA, BI, BA, DSS, DSS**. Mô phỏng dùng mô hình BA nhưng được
đặt trong quy trình so sánh phương án của DSS, vì vậy chấp nhận BA nếu giải
thích rõ góc nhìn.
</details>

## Bài tập 2 — Từ tín hiệu dữ liệu đến câu hỏi quyết định

1. Tính số khách, số rời mạng, churn rate và doanh thu tháng theo phân khúc.
2. Xác định phân khúc cần chú ý trước.
3. Viết một câu hỏi quyết định cụ thể theo mẫu: **đối tượng – hành động – mục tiêu – ràng buộc**.

In [ ]:
segment_kpi = (customers.groupby("segment", as_index=False)
               .agg(customers=("customer_id", "nunique"),
                    churners=("churned", "sum"),
                    churn_rate=("churned", "mean"),
                    monthly_revenue=("monthly_fee", "sum")))
segment_kpi["churn_rate"] = segment_kpi["churn_rate"].round(4)
segment_kpi.sort_values("churn_rate", ascending=False)

In [ ]:
ax = segment_kpi.sort_values("churn_rate").plot.barh(
    x="segment", y="churn_rate", legend=False, color="#4472C4", figsize=(7, 3.5))
ax.set(title="Tỷ lệ churn theo phân khúc", xlabel="Churn rate", ylabel="Phân khúc")
ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
plt.show()

**Ô trả lời của sinh viên:**

- Tín hiệu nổi bật: ...
- Câu hỏi quyết định: ...
- KPI đánh giá quyết định tốt hơn: ...
- Dữ liệu còn thiếu cần thu thập: ...

## Bài tập 3 — Chi phí của false positive và false negative

Quy tắc thử nghiệm: khách có `risk_score >= threshold` sẽ nhận ưu đãi.

- **False positive (FP):** ưu đãi người thực tế không rời mạng → tốn chi phí.
- **False negative (FN):** bỏ sót người thực tế rời mạng → mất giá trị khách hàng.

Giả sử ưu đãi tốn 120.000đ; giữ thành công 35% khách sắp rời mạng; giá trị giữ
được là 1.200.000đ/người. Hãy so sánh nhiều ngưỡng.

In [ ]:
OFFER_COST = 120_000
SAVE_RATE = 0.35
CUSTOMER_VALUE = 1_200_000

def evaluate_threshold(df, threshold):
    targeted = df["risk_score"] >= threshold
    actual = df["churned"].eq(1)
    tp = int((targeted & actual).sum())
    fp = int((targeted & ~actual).sum())
    fn = int((~targeted & actual).sum())
    n_targeted = int(targeted.sum())
    expected_saved = tp * SAVE_RATE
    net_benefit = expected_saved * CUSTOMER_VALUE - n_targeted * OFFER_COST
    return {"threshold": threshold, "targeted": n_targeted, "TP": tp,
            "FP": fp, "FN": fn, "expected_saved": expected_saved,
            "net_benefit": net_benefit}

threshold_results = pd.DataFrame(
    [evaluate_threshold(customers, t) for t in np.arange(0.10, 0.61, 0.05)]
)
threshold_results.sort_values("net_benefit", ascending=False).head()

In [ ]:
best = threshold_results.loc[threshold_results["net_benefit"].idxmax()]
print(f"Ngưỡng có lợi ích kỳ vọng cao nhất: {best['threshold']:.2f}")
print(f"Số người nhận ưu đãi: {best['targeted']:.0f}")
print(f"Lợi ích ròng kỳ vọng: {best['net_benefit']:,.0f} đ")

**Câu hỏi thảo luận:** Ngưỡng tối ưu có thay đổi nếu chi phí ưu đãi tăng gấp
đôi? Nếu doanh nghiệp ưu tiên trải nghiệm khách hàng hơn lợi ích ngắn hạn, chỉ
dùng `net_benefit` có đủ không?

## Bài tập 4 — Chu trình bốn pha của Simon và DSS đơn giản

Doanh nghiệp cân nhắc ba gói. Điểm phù hợp có trọng số là một mô hình hỗ trợ
so sánh, không thay thế trách nhiệm của người quản lý.

In [ ]:
options = pd.DataFrame({
    "option": ["Giảm giá trực tiếp", "Tặng dung lượng", "CSKH cá nhân hóa"],
    "cost_score": [5, 8, 7],          # cao = tiết kiệm hơn
    "benefit_score": [8, 6, 7],       # cao = lợi ích hơn
    "risk_score": [5, 7, 8],          # cao = ít rủi ro hơn
    "speed_score": [9, 8, 5]
})
weights = {"cost_score": 0.25, "benefit_score": 0.40,
           "risk_score": 0.20, "speed_score": 0.15}
options["weighted_score"] = sum(options[c] * w for c, w in weights.items())
options.sort_values("weighted_score", ascending=False)

In [ ]:
def rank_options(benefit_weight):
    remaining = 1 - benefit_weight
    w = {"benefit_score": benefit_weight, "cost_score": remaining*0.42,
         "risk_score": remaining*0.33, "speed_score": remaining*0.25}
    out = options[["option"]].copy()
    out["score"] = sum(options[c] * value for c, value in w.items())
    return out.sort_values("score", ascending=False)

rank_options(benefit_weight=0.55)

Hoàn thiện bảng bốn pha:

| Pha | Việc cần làm trong tình huống churn |
|---|---|
| 1. Nhận diện | KPI nào lệch? ở phân khúc nào? |
| 2. Thiết kế | ... |
| 3. Lựa chọn | ... |
| 4. Thực hiện & học hỏi | ... |

**Phân tích độ nhạy:** thay `benefit_weight` từ 0.3 đến 0.7. Nếu thứ hạng đổi,
điều đó nói gì về tính chắc chắn của khuyến nghị?

## Bài tập 5 — Mức độ tự động hoá và trách nhiệm

Phân loại các quyết định sau thành: `Con người quyết định`, `Bán tự động`, hoặc
`Tự động có giám sát`: gửi cảnh báo churn; cấp ưu đãi nhỏ theo luật; từ chối
tín dụng; điều chuyển tồn kho; chấm dứt hợp đồng.

Với mỗi quyết định, giải thích theo 5 tiêu chí: tính lặp lại, độ rõ của tiêu
chí, độ ổn định dữ liệu, hậu quả khi sai, và khả năng truy vết.

In [ ]:
automation_exercise = pd.DataFrame({
    "decision": ["Gửi cảnh báo churn", "Cấp ưu đãi nhỏ", "Từ chối tín dụng",
                 "Điều chuyển tồn kho", "Chấm dứt hợp đồng"],
    "your_level": [""]*5,
    "reason_and_control": [""]*5
})
automation_exercise

## Bài tập tổng hợp

Viết một **decision memo 150–200 từ** cho giám đốc:

1. Vấn đề và bằng chứng chính.
2. Phương án được đề xuất, mục tiêu và ràng buộc.
3. Một rủi ro/sai số quan trọng.
4. Kế hoạch thực hiện, A/B test và KPI giám sát.

Không được kết luận quan hệ nhân quả chỉ từ các thống kê mô tả trong notebook.